<a href="https://colab.research.google.com/github/syahidmid/google-colab/blob/main/wordpress/case-study/get_dulicate_title.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Get Posts + Extract H1 - Bali Touristic WordPress API
Notebook ini mengambil semua **posts** dari `balitouristic.com`, lalu mengekstrak semua tag `<h1>` dari field `content`.

**Tujuan:** Mengidentifikasi duplicate title antara field `title` dan `H1` di dalam `content`.

**Output CSV:** `ID`, `URL`, `Title (field)`, `H1 from Content`

In [ ]:
import requests
import csv
import json
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# Jika menggunakan Google Colab
try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    userdata = None

print("✅ Library berhasil diimport.")

In [ ]:
# Endpoint Posts
wordpress_url = "https://www.balitouristic.com/wp-json/wp/v2/posts"

# Ambil kredensial
if userdata:
    username = userdata.get('admin_bali')
    password = userdata.get('pass_bali')
else:
    username = None
    password = None

if not username or not password:
    raise ValueError("Kredensial tidak ditemukan di userdata.")

auth = HTTPBasicAuth(username, password)
print("✅ Kredensial berhasil dimuat.")

In [ ]:
# ─────────────────────────────────────────────
# Fungsi: Ambil semua posts dari WordPress API
# ─────────────────────────────────────────────
def fetch_all_posts(url, auth):
    all_posts = []
    page = 1

    print("🚀 Memulai pengambilan posts...\n")

    while True:
        print(f"📄 Mengambil halaman {page} ...")

        response = requests.get(
            url,
            auth=auth,
            params={
                "per_page": 100,
                "page": page,
                "_fields": "id,slug,status,title,content,link"
            }
        )

        if response.status_code == 200:
            posts = response.json()

            if not posts:
                print("✅ Tidak ada data lagi. Selesai.\n")
                break

            print(f"   ➜ Ditemukan {len(posts)} post di halaman {page}")
            all_posts.extend(posts)
            page += 1

        elif response.status_code == 400:
            print("✅ Semua halaman telah diambil. Selesai.\n")
            break

        else:
            print(f"❌ Gagal di halaman {page}")
            print("Status Code:", response.status_code)
            print(response.text)
            break

    print(f"\n🎯 Total post berhasil diambil: {len(all_posts)}\n")
    return all_posts

In [ ]:
# ─────────────────────────────────────────────
# Fungsi: Ekstrak semua H1 dari HTML content
# ─────────────────────────────────────────────
def extract_h1_tags(html_content):
    """
    Mengekstrak semua teks <h1> dari HTML string.
    Return: list of H1 strings (bisa kosong jika tidak ada H1)
    """
    if not html_content:
        return []
    soup = BeautifulSoup(html_content, "html.parser")
    h1_tags = soup.find_all("h1")
    return [h1.get_text(strip=True) for h1 in h1_tags]

In [ ]:
# Ambil semua posts
posts = fetch_all_posts(wordpress_url, auth)

In [ ]:
# ─────────────────────────────────────────────
# Proses: Filter posts yang punya H1 di content
# ─────────────────────────────────────────────
results = []
total_with_h1 = 0
total_without_h1 = 0

print("🔄 Memproses content untuk mencari H1 tags...\n")

for post in posts:
    post_id   = post.get("id", "")
    post_url  = post.get("link", "")
    post_title = post["title"]["rendered"] if "title" in post else ""

    # Ambil raw HTML dari content
    content_html = post.get("content", {}).get("rendered", "") if isinstance(post.get("content"), dict) else ""

    # Ekstrak semua H1
    h1_list = extract_h1_tags(content_html)

    if h1_list:
        total_with_h1 += 1
        # Satu baris per H1 (jika ada lebih dari 1 H1 di content yang sama)
        for h1_text in h1_list:
            results.append({
                "ID": post_id,
                "URL": post_url,
                "Title (field)": post_title,
                "H1 from Content": h1_text,
                "H1 Count": len(h1_list)  # info bonus: berapa total H1 di post ini
            })
    else:
        total_without_h1 += 1

print(f"📊 Statistik:")
print(f"   ✅ Post dengan H1  : {total_with_h1}")
print(f"   ⬜ Post tanpa H1   : {total_without_h1}")
print(f"   📝 Total baris CSV  : {len(results)}")

In [ ]:
# ─────────────────────────────────────────────
# Preview: 5 data pertama
# ─────────────────────────────────────────────
print("🔍 Preview 5 data pertama:\n")
for item in results[:5]:
    print(f"  ID         : {item['ID']}")
    print(f"  URL        : {item['URL']}")
    print(f"  Title      : {item['Title (field)']}")
    print(f"  H1 Content : {item['H1 from Content']}")
    print(f"  H1 Count   : {item['H1 Count']}")
    print("-" * 60)

In [ ]:
# ─────────────────────────────────────────────
# Simpan ke CSV
# ─────────────────────────────────────────────
if results:
    csv_filename = "posts_with_h1.csv"

    with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["ID", "URL", "Title (field)", "H1 from Content", "H1 Count"])
        writer.writeheader()
        writer.writerows(results)

    print(f"💾 File CSV berhasil disimpan sebagai '{csv_filename}'")
    print(f"   Total baris (termasuk header): {len(results) + 1}")

    if IN_COLAB:
        files.download(csv_filename)
        print("⬇️  File sedang didownload...")
else:
    print("⚠ Tidak ada post dengan H1 yang ditemukan.")

In [ ]:
# ─────────────────────────────────────────────
# BONUS: Identifikasi potensi duplicate title
# (Title field == H1 content, case-insensitive)
# ─────────────────────────────────────────────
duplicates = [
    r for r in results
    if r["Title (field)"].strip().lower() == r["H1 from Content"].strip().lower()
]

print(f"\n🔁 Post dengan Title == H1 (potential duplicate): {len(duplicates)}")
if duplicates:
    print("\nContoh 5 pertama:")
    for d in duplicates[:5]:
        print(f"  [{d['ID']}] {d['Title (field)']}")
        print(f"         URL: {d['URL']}")
        print("-" * 60)